In [ ]:
import json
import logging
import shutil
import time

import networkx as nx

from sysml import config, nl
from sysml.pipeline import analogy, examples, extract, load, structure

logging.disable(logging.INFO)  # every service narrates each call; keep the summaries

sources = sorted(config.MODELS.rglob("*.sysml"))
chars = sum(len(p.read_text(encoding="utf-8", errors="replace")) for p in sources)

print(f"database    {config.DB_NAME}  at {config.ARANGO_URL}")
print(f"models      {len(config.MODEL_NAMES)}  ->  {', '.join(config.MODEL_NAMES)}")
print(f"sources     {len(sources)} .sysml files, {chars:,} characters")
print(f"workbench   {config.KG}")
print(f"chat        {config.CHAT_MODEL}    embeddings  {config.EMBED_MODEL} ({config.EMBED_DIM}d)")
print(f"ontology    {len(config.KINDS)} entity types, {len(config.RELATIONS_ONTOLOGY)} relation types, closed")

In [ ]:
COLD = True  # False keeps out/kg, so extraction replays from cache and costs nothing

cleared = []
if COLD and config.KG.exists():
    cleared += [f"out/kg/{d.name}" for d in sorted(config.KG.iterdir()) if d.is_dir()]
    shutil.rmtree(config.KG)
for path in (config.OUT / "structure.json", config.AQL_EXAMPLES_GENERATED):
    if path.exists():
        path.unlink()
        cleared.append(path.name)

dropped = config.drop_database()

print(f"dropped database   {config.DB_NAME}" if dropped else f"no database {config.DB_NAME} to drop")
for name in cleared:
    print(f"removed            {name}")
if not cleared:
    print("removed            nothing")
print()
if COLD:
    print(f"COLD  -- every LLM answer is bought again: ~800 calls on {config.CHAT_MODEL}")
    print("         for this corpus, plus embeddings for entities, chunks and reports.")
else:
    print("WARM  -- out/kg kept, so extraction replays from cache and costs nothing.")

In [ ]:
# `extract.main()` wraps this in asyncio.run, which a notebook kernel already
# inside a running loop refuses. Same work, awaited.
started = time.time()
for model in config.MODEL_NAMES:
    counts = await extract.run(model)
    print(f"  extracted into {config.kg(model)}")
    for name, n in counts.items():
        print(f"  {n:>6}  {name}")
elapsed = time.time() - started

print("\n" + "=" * 78)
print(f"STEP 1  extract   {elapsed:.0f}s   ->  out/kg, a workbench on disk")
print("=" * 78)
print("The LLM read the source text and reported what it found. Nothing has")
print("touched ArangoDB yet -- this step writes GraphML and JSON to disk.")

kinds = {k.lower() for k in config.KINDS}
allowed = {r.lower() for r in config.RELATIONS_ONTOLOGY}
seen, off_ontology = {}, 0

for model in config.MODEL_NAMES:
    g = nx.read_graphml(config.kg(model) / config.ARTIFACTS.RELATIONSHIPS)
    reports = json.loads(
        (config.kg(model) / config.ARTIFACTS.COMMUNITY_REPORTS).read_text(encoding="utf-8"))
    print(f"\n  {model}")
    print(f"    {g.number_of_nodes():>5} entities  {g.number_of_edges():>5} relations  "
          f"{len(reports):>4} community reports")
    for _, d in g.nodes(data=True):
        if (d.get("entity_type") or "").lower() not in kinds:
            off_ontology += 1
    for _, _, d in g.edges(data=True):
        t = (d.get("relationship_type") or "none").lower()
        seen[t] = seen.get(t, 0) + 1
        if t not in allowed:
            off_ontology += 1

print(f"\n  ontology conformance: {len(seen)} of {len(config.RELATIONS_ONTOLOGY)} relation types used, "
      f"{off_ontology} off-ontology")
print("  WHY: enable_strict_types makes the two lists a closed vocabulary -- an entity")
print("       or edge typed outside them is dropped, not renamed. That is the whole")
print("       of what this project tells the LLM about SysML.")

In [ ]:
g = nx.read_graphml(config.kg(config.MODEL_NAMES[-1]) / config.ARTIFACTS.RELATIONSHIPS)

node, data = next((n, d) for n, d in g.nodes(data=True) if d.get("description"))
print("SAMPLE ENTITY, exactly as extraction wrote it")
print(f"  name         {node}")
print(f"  entity_type  {data.get('entity_type')}")
print(f"  clusters     {str(data.get('clusters'))[:90]}")
print(f"  description  {(data.get('description') or '')[:220]}")
print("  WHY: this is a reading, not a parse. There is no value, no unit, no file")
print("       and no line -- which is exactly what step 3 adds.")

src, dst, edata = next((a, b, d) for a, b, d in g.edges(data=True) if d.get("relationship_type"))
print("\nSAMPLE RELATION")
print(f"  {src}  --[{edata.get('relationship_type')}]->  {dst}")
print(f"  description  {(edata.get('description') or '')[:200]}")

reports = json.loads(
    (config.kg(config.MODEL_NAMES[-1]) / config.ARTIFACTS.COMMUNITY_REPORTS).read_text(encoding="utf-8"))
level0 = [r for r in reports.values() if r.get("level") == 0] or list(reports.values())
report = max(level0, key=lambda r: len(r.get("report_string") or ""))
print("\nSAMPLE COMMUNITY REPORT  (Leiden clustered the graph, then one LLM call per cluster)")
print(f"  title   {report.get('title')}")
print(f"  level   {report.get('level')}   sub-communities {len(report.get('sub_communities') or [])}")
print("  " + (report.get("report_string") or "")[:420].replace("\n", "\n  "))
print("\n  WHY: these reports are what the `global` retriever answers from. No single")
print("       row says what the corpus is about; a community report does.")

In [ ]:
# Same reason as step 1: load.main() calls asyncio.run internally.
started = time.time()
counts = await load.load()
elapsed = time.time() - started
print(f"  loaded into {config.DB_NAME}")
for name, n in counts.items():
    print(f"  {n:>6}  {name}")

db = config.db()

print("\n" + "=" * 78)
print(f"STEP 2  load   {elapsed:.0f}s   ->  ArangoDB, via the importer's own writer")
print("=" * 78)
print("ImportGraphToADB is the class the platform's importer pods run. Using it")
print("rather than imitating its schema means the graph is the shape a")
print("platform-built graph has by construction -- which is what the retrievers")
print("and the AQLizer were written to read.")

print("\n  collections")
for name in config.ALL_COLLECTIONS:
    print(f"    {db.collection(name).count():>7}  {name}")

print("\n  edge kinds")
for row in db.aql.execute(f"""
        FOR r IN {config.RELATIONS}
          COLLECT t = r.type WITH COUNT INTO n
          SORT n DESC RETURN {{t, n}}"""):
    print(f"    {row['n']:>7}  {row['t']}")

In [ ]:
db = config.db()

print("SAMPLE DOCUMENT")
doc = next(iter(db.aql.execute(f"FOR d IN {config.DOCUMENTS} LIMIT 1 RETURN d")))
for key in ("_key", "file_name", "citable_url", "file_ids", "models"):
    if key in doc:
        print(f"  {key:<13} {str(doc[key])[:90]}")

print("\nSAMPLE CHUNK")
chunk = next(iter(db.aql.execute(f"FOR c IN {config.CHUNKS} LIMIT 1 RETURN c")))
print(f"  _key          {chunk['_key']}")
print(f"  files         {chunk.get('files')}")
print(f"  content       {(chunk.get('content') or '')[:180]!r}")
print(f"  embedding     [{len(chunk.get(config.EMBEDDING_FIELD) or [])} floats]")

print("\nPROVENANCE CHAIN  entity -> chunk -> document, built by the importer")
for row in db.aql.execute(f"""
        FOR e IN {config.ENTITIES}
          FILTER e.source_file != null
          LET chain = FIRST(
            FOR chunk IN 1..1 OUTBOUND e {config.RELATIONS}
              FILTER IS_SAME_COLLECTION('{config.CHUNKS}', chunk)
              FOR d IN 1..1 OUTBOUND chunk {config.RELATIONS}
                FILTER IS_SAME_COLLECTION('{config.DOCUMENTS}', d)
                RETURN {{chunk: chunk._key, document: d.file_name}})
          FILTER chain != null
          LIMIT 3
          RETURN {{entity: e.entity_name, chunk: chain.chunk, document: chain.document}}"""):
    print(f"  {row['entity']:<34} -> {row['chunk']:<24} -> {row['document']}")

print("\n  WHY: every answer can name the file it came from. `load` then resolves this")
print("       two-hop walk once and writes `files` and `models` onto every row, so an")
print("       analytical question is a filter instead of a traversal.")

In [ ]:
db = config.db()

print("=" * 78)
print("STEP 3  structure   ->  the same files read again, with a lexer")
print("=" * 78)
print("This ran inside `load`, before the vector indexes were built. It writes down")
print("only what the syntax states outright, and marks all of it `stated: true`.")

q = lambda aql: next(iter(db.aql.execute(aql)))
print("\n  entities")
print(f"    {q(f'RETURN LENGTH(FOR e IN {config.ENTITIES} FILTER e.source_file != null RETURN 1)'):>7}"
      "  carry a source file and line  (declared)")
print(f"    {q(f'RETURN LENGTH(FOR e IN {config.ENTITIES} FILTER e.source_file == null RETURN 1)'):>7}"
      "  do not  (the LLM inferred them)")
print(f"    {q(f'RETURN LENGTH(FOR e IN {config.ENTITIES} FILTER LENGTH(ATTRIBUTES(e.attributes)) > 0 RETURN 1)'):>7}"
      "  carry at least one typed attribute")
print(f"    {q(f'RETURN LENGTH(FOR e IN {config.ENTITIES} FILTER e.short_name != null RETURN 1)'):>7}"
      "  carry a short name")

print("\n  relations")
print(f"    {q(f'RETURN LENGTH(FOR r IN {config.RELATIONS} FILTER r.stated == true RETURN 1)'):>7}"
      "  read from the syntax")
print(f"    {q(f'RETURN LENGTH(FOR r IN {config.RELATIONS} FILTER r.type == \'RELATED_TO\' AND r.stated != true RETURN 1)'):>7}"
      "  inferred by the LLM")

In [ ]:
db = config.db()

row = next(iter(db.aql.execute(f"""
    FOR e IN {config.ENTITIES}
      FILTER e.source_file != null AND LENGTH(ATTRIBUTES(e.attributes)) > 1
      SORT LENGTH(ATTRIBUTES(e.attributes)) DESC
      LIMIT 1 RETURN e""")))

print("SAMPLE ENTITY after structure -- one document, several writers")
for key, value in row.items():
    if key in ("_id", "_rev", "partition_id"):
        continue
    if key == config.EMBEDDING_FIELD:
        value = f"[{len(value)} floats]"
    text = str(value)
    print(f"  {key:<14} {text[:110] + (' ...' if len(text) > 110 else '')}")

print("\n  description / clusters / embedding  came from extraction and Leiden")
print("  files / models                      came from load's two-hop resolution")
print("  source_file / source_line / attributes / short_name   came from the lexer")

edge = next(iter(db.aql.execute(f"""
    FOR r IN {config.RELATIONS} FILTER r.stated == true LIMIT 1 RETURN r""")))
print("\nSAMPLE STATED EDGE -- identical in shape to an extracted one")
for key in ("_from", "_to", "type", "relationship_type", "description",
            "stated", "source_file", "source_line"):
    if key in edge:
        print(f"  {key:<18} {str(edge[key])[:96]}")
print("\n  WHY: nothing downstream has to know which pass wrote an edge. `stated` is")
print("       the only discriminator, and it is there for the queries that do care.")

In [ ]:
db = config.db()

result = next(iter(db.aql.execute(f"""
    FOR e IN {config.ENTITIES}
      FILTER e.entity_name == "SATURNV"
      LET parts = (
        FOR c, edge, path IN 1..2 OUTBOUND e {config.RELATIONS}
          FILTER path.edges[*].relationship_type ALL IN ["owns", "typedby"]
          FILTER c.attributes.dryMass.value != null
          RETURN DISTINCT {{name: c.entity_name, kg: c.attributes.dryMass.value,
                           unit: c.attributes.dryMass.unit,
                           at: CONCAT(c.source_file, ":", c.source_line)}})
      RETURN {{total: SUM(parts[*].kg), parts}}""")))

print("THE PAYOFF -- sum the Saturn V's dry mass by walking its containment tree")
for part in sorted(result["parts"], key=lambda p: -p["kg"]):
    print(f"  {part['kg']:>8,} {part['unit']}   {part['name']:<26} {part['at']}")
print(f"  {'-' * 8}")
print(f"  {result['total']:>8,} kg   total from {len(result['parts'])} declared contributors")
print("\n  WHY: without the lexer this returns nothing -- not because the masses are")
print("       missing, but because there is no path from the vehicle to its stages")
print("       to walk. Exact numbers on an exact tree is what makes arithmetic work.")

In [ ]:
started = time.time()
analogy.main()
elapsed = time.time() - started

print("\n" + "=" * 78)
print(f"STEP 4  analogy   {elapsed:.0f}s   ->  autograph's SimilarityFinder, unmodified")
print("=" * 78)
print("Nothing in the graph crosses a model boundary yet: no Apollo file mentions a")
print("drone, so extraction cannot relate them. This step computes the correspondence.")

print("\n  edges joining two entities with no model in common")
for row in db.aql.execute(f"""
        FOR r IN {config.RELATIONS}
          LET a = DOCUMENT(r._from), b = DOCUMENT(r._to)
          FILTER a.models != null AND b.models != null
          FILTER LENGTH(INTERSECTION(a.models, b.models)) == 0
          COLLECT kind = r.type WITH COUNT INTO n
          RETURN {{kind, n}}"""):
    print(f"    {row['n']:>5}  {row['kind']}")
print("    (SIMILAR_TO is the only one, which is the point)")

In [ ]:
db = config.db()

print("SAMPLE ANALOGIES -- strongest cross-model pairs")
for row in db.aql.execute(f"""
        FOR r IN {config.RELATIONS}
          FILTER r.type == "{config.SIMILAR_TO}"
          LET a = DOCUMENT(r._from), b = DOCUMENT(r._to)
          FILTER LENGTH(INTERSECTION(a.models, b.models)) == 0
          SORT r.cosine DESC LIMIT 12
          RETURN {{a: a.entity_name, am: a.models[0], b: b.entity_name, bm: b.models[0],
                  role: r.analogy_role, cosine: r.cosine}}"""):
    print(f"  {row['cosine']:.3f}  {row['role']:<11} {row['a']} ({row['am']})")
    print(f"         {'':<11} ~  {row['b']} ({row['bm']})")

print("\n  WHY: `module_doc_ids` exists upstream to keep similarity edges INSIDE a")
print("       module. Passing the other models' entities runs it backwards, so the")
print("       only edges it can build are the ones that cross. Entities are compared")
print("       only against their own type, or a part's nearest neighbour is reliably")
print("       a requirement that talks about the part -- topical, not analogous.")

In [ ]:
started = time.time()
examples.main()
elapsed = time.time() - started

print("\n" + "=" * 78)
print(f"STEP 5  examples   {elapsed:.0f}s   ->  the AQLizer primer, written from the graph")
print("=" * 78)

generated = config.AQL_EXAMPLES_GENERATED.read_text(encoding="utf-8")
blocks = generated.count("```aql")
print(f"  {config.AQL_EXAMPLES_GENERATED.name}")
print(f"  {len(generated):,} characters, {blocks} worked AQL examples")
print("\n  Every one of those blocks was executed against the finished graph and any")
print("  that failed went back to the model with its error for one repair round.")
print("  WHY: the chain copies the SHAPE of a worked example. One that does not run")
print("       is worse than no example at all.")

print("\n  " + "-" * 74)
print("  " + "\n  ".join(generated.splitlines()[:38]))

In [ ]:
import matplotlib.pyplot as plt

db = config.db()

edge_rows = list(db.aql.execute(f"""
    FOR r IN {config.RELATIONS} COLLECT t = r.type WITH COUNT INTO n
    SORT n DESC RETURN {{t, n}}"""))
declared = next(iter(db.aql.execute(
    f"RETURN LENGTH(FOR e IN {config.ENTITIES} FILTER e.source_file != null RETURN 1)")))
inferred = next(iter(db.aql.execute(
    f"RETURN LENGTH(FOR e IN {config.ENTITIES} FILTER e.source_file == null RETURN 1)")))
stated = next(iter(db.aql.execute(
    f"RETURN LENGTH(FOR r IN {config.RELATIONS} FILTER r.stated == true RETURN 1)")))
guessed = next(iter(db.aql.execute(
    f"RETURN LENGTH(FOR r IN {config.RELATIONS} FILTER r.type == 'RELATED_TO' AND r.stated != true RETURN 1)")))

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.2))

names = [r["t"] for r in edge_rows][::-1]
counts = [r["n"] for r in edge_rows][::-1]
bars = left.barh(names, counts, color="#4C72B0")
if config.SIMILAR_TO in names:
    bars[names.index(config.SIMILAR_TO)].set_color("#C44E52")
left.set_title(f"{sum(counts):,} edges, by kind")
left.bar_label(bars, fmt="%d", padding=3, fontsize=9)
left.set_xlim(0, max(counts) * 1.18)
for side in ("top", "right"):
    left.spines[side].set_visible(False)

right.barh(["entities", "relations"], [declared, stated], color="#55A868", label="from the syntax")
right.barh(["entities", "relations"], [inferred, guessed], left=[declared, stated],
           color="#CCB974", label="from the LLM")
right.set_title("provenance: what the files state vs what was inferred")
right.legend(loc="lower right", frameon=False)
for side in ("top", "right"):
    right.spines[side].set_visible(False)
for y, (a, b) in enumerate(((declared, inferred), (stated, guessed))):
    right.text(a / 2, y, f"{a:,}", ha="center", va="center", color="white", fontsize=10)
    right.text(a + b / 2, y, f"{b:,}", ha="center", va="center", fontsize=10)

plt.tight_layout()
plt.show()

print(f"{len(config.MODEL_NAMES)} models  ->  "
      f"{db.collection(config.DOCUMENTS).count()} documents, "
      f"{db.collection(config.CHUNKS).count()} chunks, "
      f"{db.collection(config.ENTITIES).count():,} entities, "
      f"{db.collection(config.COMMUNITIES).count()} communities, "
      f"{db.collection(config.RELATIONS).count():,} edges")
print("Filter on source_file != null and stated == true and you are querying the")
print("model the engineers wrote. Do not, and you are querying that plus inference.")

In [ ]:
nl.instance().ask(
    "For each Saturn V stage, give its dry mass, its propellant mass and the sum of "
    "the two, sorted by the total, with the file and line each is declared on."
).show(row_limit=7)

In [ ]:
nl.instance().ask(
    "What did the Apollo 11 mission cost in total, and what are the parts of that "
    "figure?").show()

In [ ]:
nl.instance().ask(
    "Which ten elements have the largest mass or dry mass, with units and the "
    "file and line each is declared on?").show(row_limit=10)

In [ ]:
nl.instance().ask(
    "Which ten Apollo requirements have the most elements satisfying them, and how "
    "many Apollo requirements have none at all?"
).show(row_limit=3)

In [ ]:
nl.instance().ask(
    "Trace HLR-R001: what satisfies it, what refines it, and what it is related "
    "to in either direction."
).show(row_limit=8)

In [ ]:
nl.instance().ask(
    "Break the relations down by model and by whether they were read from the "
    "syntax or inferred by the LLM."
).show(row_limit=6)

In [ ]:
nl.instance().ask(
    "Which requirements does the drone model state that the Apollo model has an "
    "analogous requirement for, and how close are they?"
).show(row_limit=6)

In [ ]:
nl.instance().ask(
    "What does the drone's powerManagementModule correspond to in the Apollo model?"
).show()

In [ ]:
answer = nl.instance(config.AQL_EXAMPLES_GENERATED).ask(
    "Which requirements in the DroneModelLogical model does nothing satisfy?")
answer.show(row_limit=8)
print(f"\nprimed with  {config.AQL_EXAMPLES_GENERATED.name}  (written by step 5, not by hand)")

In [ ]:
(await nl.retriever().ask_async(
    "What keeps the astronauts alive and breathing, and what limits does it "
    "have to hold?"
)).show(row_limit=4)

In [ ]:
answer = await nl.retriever().ask_async(
    "How much drinking water must the environmental control system supply "
    "per crew member per day?",
    scope="unified")
answer.show(row_limit=3)
answer.evidence(chars=420, find="water")

In [ ]:
answer = await nl.retriever().ask_async(
    "What is the minimum delta-v the lunar module ascent stage has to provide, "
    "and why that figure?",
    scope="unified")
answer.show(row_limit=3)
answer.evidence(chars=320, find="delta")

In [ ]:
(await nl.retriever().ask_async(
    "What concerns are these models organised around, and what does each part "
    "of the corpus contribute?",
    scope="global")).show()

In [ ]:
(await nl.retriever().ask_async(
    "What in the Apollo model plays a role like the drone's power management module?"
)).show(row_limit=4)

In [ ]:
QUESTION = "How many Apollo requirements does nothing satisfy?"

nl.instance().ask(QUESTION).show(row_limit=2)

In [ ]:
(await nl.retriever().ask_async(QUESTION)).show(row_limit=3)